[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Digital-AI-Finance/Introduction-to-Machine-Learning-notebooks/blob/master/block2.ipynb)

# Block 2: fitting a model, and finding out whether it worked

Work through this before block 2, on your own or in pairs. It takes about
forty minutes. Every answer is written underneath the question, so you cannot
get stuck.

If a cell fails, read the message, then run the cell above it again.

## 1. A real table that ships with scikit-learn

569 rows. Each row is one sample, each column is one measurement, and the
label says which of two kinds it is. Nothing is downloaded: this table is
inside the scikit-learn you already have.

In [ ]:
import pandas as pd
from sklearn.datasets import load_breast_cancer

raw = load_breast_cancer()
X = pd.DataFrame(raw.data, columns=raw.feature_names)
y = raw.target

print("rows and columns:", X.shape)
print("labels: ", pd.Series(y).value_counts().to_dict())
X.iloc[:5, :4]

## 2. Hide some of it before you start

Seven rows in ten to learn from, three to check on. `random_state` fixes the
shuffle so everybody gets the same split.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=0)

print("learning from:", X_train.shape[0], "rows")
print("held back:    ", X_test.shape[0], "rows")

## 3. Two models, fitted

A logistic regression, which is the line that answers yes or no, and a
decision tree. Three lines each.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

line = LogisticRegression(max_iter=5000).fit(X_train, y_train)
tree = DecisionTreeClassifier(random_state=0).fit(X_train, y_train)

for name, m in [("logistic regression", line), ("decision tree", tree)]:
    print("%-20s learned from %.3f   held back %.3f" % (
        name, m.score(X_train, y_train), m.score(X_test, y_test)))

Look at the tree's two numbers. It is perfect on the rows it learned from and
worse on the rows it never saw. That gap is overfitting.

## 4. All four numbers

Accuracy is one number and it hides which mistake was made. The four numbers
say it properly.

In [ ]:
from sklearn.metrics import confusion_matrix

for name, m in [("logistic regression", line), ("decision tree", tree)]:
    tn, fp, fn, tp = confusion_matrix(y_test, m.predict(X_test)).ravel()
    print(name)
    print("   said yes and was right:", tp, "   said yes and was wrong:", fp)
    print("   said no and was right: ", tn, "   said no and was wrong: ", fn)

The bottom right of each is the expensive one here: cases it called healthy
that were not. Accuracy alone never showed you that number.

## 5. What if you hid the wrong rows?

One split gives one score, and that score depends on which rows you happened
to hide. Five folds gives five scores.

In [ ]:
from sklearn.model_selection import cross_val_score

for name, m in [("logistic regression", LogisticRegression(max_iter=5000)),
                ("decision tree", DecisionTreeClassifier(random_state=0))]:
    scores = cross_val_score(m, X, y, cv=5)
    print("%-20s %s" % (name, " ".join("%.3f" % s for s in scores)))
    print("%-20s lowest %.3f, highest %.3f" % ("", scores.min(), scores.max()))

Read the two rows for the tree. The average is respectable and the lowest and
highest are far apart, which means the answer moves depending on what you
hid. One lucky split would have told you the higher number.

That spread is the thing a single score cannot show you.

## 6. Twenty trees, and a vote

One tree overfits. Twenty trees, each grown on a different part of the data,
voting on the answer, is a random forest.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

forest = RandomForestClassifier(n_estimators=20, random_state=0)
scores = cross_val_score(forest, X, y, cv=5)

print("random forest ", " ".join("%.3f" % s for s in scores))
print("              lowest %.3f, highest %.3f" % (scores.min(), scores.max()))

Better than the single tree on the average, and the five scores sit closer
together. That is the ensemble doing its job: the trees make different
mistakes, and the vote cancels most of them.

## 7. Your turn

Change `n_estimators=20` to `n_estimators=200` and run the cell again. Then
change `test_size=0.3` to `test_size=0.5` in section 2 and run everything
below it.

Two things to notice. More trees helps and then stops helping. And holding
back half the data leaves less to learn from, so the scores drop.

Bring one number you did not expect to block 2.